# ILI9488 TFT LCD Display Tutorial

This tutorial demonstrates how to use the ILI9488 TFT LCD display driver with STM32F429I-DISC1.

## Overview

The ILI9488 is a popular TFT LCD controller supporting 320x480 resolution displays. This driver provides:
- Full color graphics (RGB565)
- Text rendering
- Basic shapes (lines, rectangles, circles)
- Multiple orientations
- SPI interface communication

## Hardware Setup

### Required Connections

| ILI9488 Pin | STM32F429 Pin | Function |
|-------------|---------------|----------|
| VCC | 3.3V | Power supply |
| GND | GND | Ground |
| CS | PB12 | Chip select |
| RESET | PB13 | Reset (active low) |
| DC/RS | PB14 | Data/Command |
| MOSI | PF9 | SPI data (SPI5) |
| SCK | PF7 | SPI clock (SPI5) |
| LED | 3.3V | Backlight |
| MISO | PF8 | SPI data (optional, SPI5) |

## Software Setup

### Enable the Driver

In your CMakeLists.txt, enable the ILI9488 driver:

```cmake
option(USE_ILI9488 "Include ILI9488 high-res TFT display driver" ON)
```

Or enable it during cmake configuration:
```bash
cmake -DUSE_ILI9488=ON ..
```

## Basic Usage

### Include Headers

```c
#include "ili9488.h"
// SPI is handled by the central SPI driver
```

### Initialize the Display

```c
// Declare display handle
ILI9488_Handle_t hili;

// Initialize with SPI5, pins PB12(CS), PB13(DC), PB14(RST)
ILI9488_StatusTypeDef status = ILI9488_Init(&hili,
                                           GPIOB, GPIO_PIN_12,  // CS
                                           GPIOB, GPIO_PIN_13,  // DC
                                           GPIOB, GPIO_PIN_14); // RST

if (status != ILI9488_OK) {
    // Handle initialization error
}
```

### Basic Display Operations

#### Clear Screen
```c
// Clear screen with black color
ILI9488_Clear(&hili, ILI9488_COLOR_BLACK);

// Clear with other colors
ILI9488_Clear(&hili, ILI9488_COLOR_WHITE);
ILI9488_Clear(&hili, ILI9488_COLOR_RED);
ILI9488_Clear(&hili, ILI9488_COLOR_BLUE);
```

#### Draw Pixels
```c
// Draw single pixel
ILI9488_DrawPixel(&hili, 100, 100, ILI9488_COLOR_RED);

// Draw multiple pixels
for (int x = 50; x < 150; x++) {
    for (int y = 50; y < 150; y++) {
        ILI9488_DrawPixel(&hili, x, y, ILI9488_COLOR_GREEN);
    }
}
```

#### Draw Shapes
```c
// Draw rectangle (outline)
ILI9488_DrawRectangle(&hili, 50, 50, 100, 100, ILI9488_COLOR_BLUE);

// Draw filled rectangle
ILI9488_DrawFilledRectangle(&hili, 200, 50, 100, 100, ILI9488_COLOR_YELLOW);

// Draw circle (outline)
ILI9488_DrawCircle(&hili, 160, 120, 50, ILI9488_COLOR_MAGENTA);

// Draw filled circle
ILI9488_DrawFilledCircle(&hili, 160, 240, 30, ILI9488_COLOR_CYAN);

// Draw line
ILI9488_DrawLine(&hili, 0, 0, 319, 479, ILI9488_COLOR_WHITE);
```

#### Text Rendering
```c
// Set cursor position
ILI9488_SetCursor(&hili, 10, 10);

// Write single character
ILI9488_WriteChar(&hili, 'A', ILI9488_COLOR_WHITE, ILI9488_COLOR_BLACK);

// Write string
ILI9488_WriteString(&hili, "Hello World!", ILI9488_COLOR_WHITE, ILI9488_COLOR_BLACK);

// Write at specific position
ILI9488_SetCursor(&hili, 50, 200);
ILI9488_WriteString(&hili, "ILI9488 Display", ILI9488_COLOR_RED, ILI9488_COLOR_BLACK);
```

### Orientation Control

```c
// Set portrait mode (default)
ILI9488_SetOrientation(&hili, ILI9488_ORIENTATION_PORTRAIT);

// Set landscape mode
ILI9488_SetOrientation(&hili, ILI9488_ORIENTATION_LANDSCAPE);

// Set rotated orientations
ILI9488_SetOrientation(&hili, ILI9488_ORIENTATION_PORTRAIT_REV);
ILI9488_SetOrientation(&hili, ILI9488_ORIENTATION_LANDSCAPE_REV);
```

### Display Control

```c
// Turn display on
ILI9488_DisplayOn(&hili);

// Turn display off (saves power)
ILI9488_DisplayOff(&hili);

// Update screen (if using buffered mode)
ILI9488_UpdateScreen(&hili);
```

## Complete Example

```c
#include "ili9488.h"
#include "spi.h"

ILI9488_Handle_t hili;

void ILI9488_Demo(void) {
    // Initialize display
    ILI9488_Init(&hili, &hspi1, GPIOB, GPIO_PIN_12, GPIOB, GPIO_PIN_13, GPIOB, GPIO_PIN_14);
    
    // Clear screen
    ILI9488_Clear(&hili, ILI9488_COLOR_BLACK);
    
    // Draw some graphics
    ILI9488_DrawFilledRectangle(&hili, 10, 10, 300, 50, ILI9488_COLOR_BLUE);
    ILI9488_DrawCircle(&hili, 160, 240, 80, ILI9488_COLOR_RED);
    
    // Write text
    ILI9488_SetCursor(&hili, 20, 25);
    ILI9488_WriteString(&hili, "ILI9488 TFT Display", ILI9488_COLOR_WHITE, ILI9488_COLOR_BLUE);
    
    ILI9488_SetCursor(&hili, 50, 300);
    ILI9488_WriteString(&hili, "320x480 Resolution", ILI9488_COLOR_YELLOW, ILI9488_COLOR_BLACK);
}
```

## Color Definitions

The driver provides predefined colors:

- `ILI9488_COLOR_BLACK`
- `ILI9488_COLOR_WHITE`
- `ILI9488_COLOR_RED`
- `ILI9488_COLOR_GREEN`
- `ILI9488_COLOR_BLUE`
- `ILI9488_COLOR_YELLOW`
- `ILI9488_COLOR_MAGENTA`
- `ILI9488_COLOR_CYAN`

You can also define custom colors using RGB565 format:
```c
#define MY_COLOR 0xF800  // Red in RGB565
```

## Performance Tips

1. **Batch Operations**: Drawing individual pixels is slow. Use filled rectangles for large areas.
2. **Minimize SPI Traffic**: Group drawing operations to reduce SPI transactions.
3. **Use Appropriate Data Types**: Coordinates are 16-bit, colors are 16-bit RGB565.
4. **Buffer Management**: For complex graphics, consider implementing a framebuffer.

## Troubleshooting

### Display Shows Nothing
- Check power connections (3.3V, GND)
- Verify SPI pin connections
- Ensure backlight is connected
- Check reset timing

### Colors Are Wrong
- Verify RGB565 color format
- Check endianness of color data
- Confirm display initialization sequence

### Touch Doesn't Work (if combined with XPT2046)
- Ensure proper SPI chip select sharing
- Check interrupt pin connections
- Verify touchscreen calibration